# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Fasiuu/FLYRANK_TASK1/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [ ]:
# Setup — shared feature builder from this repo + DuckDB connection to the hosted release.
# Token: Colab Secret HF_TOKEN (preferred) or a getpass prompt. Never paste a token into a cell.
%pip -q install duckdb
import os, sys, subprocess, json, warnings
warnings.filterwarnings('ignore')
if not os.path.exists('FLYRANK_TASK1'):
    subprocess.run(['git', 'clone', '-q', '--depth', '1', 'https://github.com/Fasiuu/FLYRANK_TASK1.git'], check=True)
sys.path.insert(0, 'FLYRANK_TASK1/work/scripts')
import numpy as np, pandas as pd
import flyrank_features as ff
pd.set_option('display.width', 200); pd.set_option('display.max_columns', 40)
np.random.seed(ff.SEED)
con = ff.connect()
os.makedirs('work/outputs', exist_ok=True)
print('connected; seed =', ff.SEED)

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

**The contract in five plain-words answers:**

1. **What one row means:** one page (`content_hash_id`) at one month-end decision date T.
2. **Which tables:** `fact_content_daily_performance` (daily Search Console numbers per page), plus `dim_content` (creation date, word count, keyword volume) and `dim_clients` (when each site's Search Console feed starts).
3. **Which time window:** features from the 90 days up to T; the outcome from the 30 days after T. Development dates are T = 2026-03-31 and 2026-04-30; June 2026 stays sealed.
4. **What I predict (label):** whether the page *falls behind its own site* next month: its impressions ratio (next 30 days vs its normal month) is below 0.8 × the same ratio for the whole site. It's a ranking target for a refresh queue, not a traffic forecast.
5. **One thing I deliberately exclude:** `content_updated_date` / `last_optimized_date`. They are export-time values, so a page edited in May already shows May when I stand on March 31.

**Time windows (all derived from `fact_content_daily_performance`):**

| window | days | used for |
|---|---|---|
| feature window | (T-90, T] | every feature |
| baseline for the label | (T-60, T] | the page's and site's normal monthly level |
| outcome window | (T, T+30] | label only |

**Dates in this notebook:** T = 2026-03-31 and 2026-04-30, so the reads cover December 2025 to May 2026. June 2026 is not touched here; it is the sealed month.

**Population:** at least 200 impressions in (T-60, T], page at least 90 days old, client GSC feed starting on or before T-89. The cell below states these claims and checks them.

In [ ]:
# Raw-table checks first (Parquet metadata is cheap), then the per-item panel.
fact = f"read_parquet({ff.month_paths('2025-12-01', '2026-05-31')})"
print(con.sql(f"SELECT MIN(report_date) first_day, MAX(report_date) last_day, COUNT(*) AS rows FROM {fact}").df().to_string(index=False))
cl = con.sql(f"SELECT client_hash_id, access_profile, gsc_data_start, ga4_data_start FROM read_parquet('{ff.REL}/dim_clients.parquet')").df()
for T in ff.DEV_DATES:
    ok = (pd.to_datetime(cl.gsc_data_start) <= pd.Timestamp(T) - pd.Timedelta(days=89)).sum()
    print(f'decision date {T}: {ok} of {len(cl)} clients have GSC history covering the full 90-day feature window')

# Build the per-item panel for the two development decision dates (one DuckDB scan, ~5-8 min).
import time
t0 = time.time()
panel = ff.build_panel(con, ff.DEV_DATES)
panel['decision_date'] = pd.to_datetime(panel['decision_date']).dt.strftime('%Y-%m-%d')
m = ff.model_frame(panel)
print(f'{len(panel):,} item-date rows built in {time.time()-t0:.0f}s; {len(m):,} pass the population checks')
print(panel.groupby('decision_date').agg(rows=('content_hash_id', 'size'), clients=('client_hash_id', 'nunique'),
      full_history=('full_history', 'sum')).to_string())
print(m.groupby('decision_date').agg(model_rows=('declined', 'size'), clients=('client_hash_id', 'nunique')).to_string())

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

Every field I plan to touch, sorted into exactly one bucket. Excluded fields each get a reason. The two exclusions that matter most:

- **`content_updated_date` / `last_optimized_date`.** These are export-time values (July 2026). If a page was edited in May, the field says May even when I'm pretending to stand on March 31. Using it would put future information into the features. I test what happens if I use it anyway in the leakage notebook.
- **`fact_content_query_90d`.** Its fixed window (2026-04-02 to 2026-06-30) overlaps every outcome month I use. Notebook 03 used its columns as features for a June label, which turns out to be leakage. I don't carry that forward.

In [ ]:
contract = pd.DataFrame([
  ('client_hash_id / content_hash_id', 'context', '-', 'join, group, split only - pseudonyms, never features'),
  ('decision_date', 'context', '-', 'which snapshot a row belongs to'),
  ('imp_last30 / imp_prev30 / imp_prev60_90 (log)', 'feature', '(T-90, T]', 'search exposure in three 30-day blocks'),
  ('imp_momentum / imp_momentum_long', 'feature', '(T-90, T]', 'log-ratio of recent vs earlier exposure'),
  ('clicks_last30 (log) / ctr_last30', 'feature', '(T-30, T]', 'demand captured; CTR in % of impressions'),
  ('pos_last30 / pos_change / pos_volatility', 'feature', '(T-60, T]', 'impression-weighted rank, its drift, its daily spread'),
  ('days_with_imp_last30 / spike_share / weekend_share', 'feature', '(T-30, T]', 'how steady the visibility is'),
  ('content_age_days', 'feature', 'at T', 'days since creation (population: >= 90)'),
  ('word_count / search_volume (log + has_ flags)', 'feature', 'static metadata', 'depth and keyword demand; blanks flagged'),
  ('imp_next30', 'label source', '(T, T+30]', 'the outcome month - never a feature'),
  ('client_ratio / page_ratio', 'label source', '(T, T+30]', 'site-level and page-level change used to define the label'),
  ('declined', 'label', '(T, T+30]', 'page_ratio < 0.8 x client_ratio'),
  ('declined_abs', 'reference label', '(T, T+30]', 'plain >20% drop - kept only to show why it was rejected'),
  ('vanished_next30', 'context', '(T, T+30]', 'no rows at all in the outcome month - reported, not modelled'),
  ('gsc_data_start', 'context', '-', 'population filter: client feed must cover the feature window'),
  ('content_updated_date / last_optimized_date', 'excluded', 'export time', 'current values; an edit after T is future information'),
  ('fact_content_query_90d (all columns)', 'excluded', 'fixed 2026-04-02 -> 06-30', 'overlaps every label month used here'),
  ('ga4_* sessions / engagement / scroll', 'excluded', '-', 'GA4 starts Feb-Mar 2026 for most clients - too little history'),
  ('sessions_ai / ai_*', 'excluded', '-', 'too sparse to be a stable feature (lane guide)'),
  ('provider_used / model_used', 'excluded', '-', 'generation metadata; lane guide: not a model feature'),
], columns=['field', 'bucket', 'window', 'why'])
print(contract.to_string(index=False))

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

**Three verification queries** on one mid-panel month (`month=2026-03`): the grain (zero duplicate date-client-page keys), the slice's row count and date span, and availability filtered with `IS TRUE`, with the number of rows that survive. The flag is three-valued, so `= FALSE` or `NOT flag` would treat NULL rows differently without saying so.

**Five features, max, from that same month**, each with its "available when?" line in the output. The label is April's falls-behind-the-site flag.

**The trap.** I add one label-derived column on purpose (the page's April impressions, which is the outcome month itself), and the quick client-grouped score jumps toward perfect. Then I delete it and keep the honest number. This is the notebook 02 lesson, run on the real warehouse.


In [ ]:
# ---- Three verification queries on one mid-panel month (month=2026-03) ----
one = f"read_parquet('{ff.REL}/fact_content_daily_performance/month=2026-03/*.parquet')"
print('QUERY 1 - grain: is one row really one (report_date, client, page)? duplicate keys, expect 0')
print(con.sql(f"""SELECT COUNT(*) AS duplicate_keys FROM (SELECT report_date, client_hash_id, content_hash_id, COUNT(*) c
                  FROM {one} GROUP BY ALL HAVING c > 1)""").df().to_string(index=False))
print('\nQUERY 2 - slice size and date span:')
print(con.sql(f"""SELECT COUNT(*) AS rows, MIN(report_date) AS first_day, MAX(report_date) AS last_day,
                  COUNT(DISTINCT client_hash_id) AS clients, COUNT(DISTINCT content_hash_id) AS pages FROM {one}""").df().to_string(index=False))
print('\nQUERY 3 - availability, filtered with IS TRUE (the flag is TRUE / FALSE / NULL), and how many rows survive:')
print(con.sql(f"""SELECT COUNT(*) AS all_rows,
                  COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) AS survive_gsc_is_true,
                  COUNT(*) FILTER (WHERE gsc_data_available IS NOT TRUE) AS dropped_false_or_null,
                  COUNT(*) FILTER (WHERE gsc_data_available IS NULL) AS of_which_null FROM {one}""").df().to_string(index=False))

# ---- Five features, max, from that same month (decision moment: 2026-03-31; outcome: April) ----
mar_apr = f"read_parquet({ff.month_paths('2026-03-01', '2026-04-30')})"
fr = con.sql(f"""
WITH d AS (SELECT client_hash_id, content_hash_id, report_date, COALESCE(gsc_impressions, 0) AS imp,
                  COALESCE(gsc_clicks, 0) AS clk, gsc_avg_position AS pos
           FROM {mar_apr} WHERE gsc_data_available IS TRUE)
SELECT client_hash_id, content_hash_id,
  SUM(imp) FILTER (WHERE report_date <= DATE '2026-03-31') AS imp_mar,
  SUM(clk) FILTER (WHERE report_date <= DATE '2026-03-31') AS clk_mar,
  SUM(pos * imp) FILTER (WHERE report_date <= DATE '2026-03-31' AND pos IS NOT NULL)
      / NULLIF(SUM(imp) FILTER (WHERE report_date <= DATE '2026-03-31' AND pos IS NOT NULL), 0) AS pos_mar,
  COUNT(*) FILTER (WHERE report_date <= DATE '2026-03-31' AND imp > 0) AS days_mar,
  MAX(imp) FILTER (WHERE report_date <= DATE '2026-03-31') AS max_day_mar,
  COALESCE(SUM(imp) FILTER (WHERE report_date > DATE '2026-03-31'), 0) AS imp_apr
FROM d GROUP BY ALL HAVING SUM(imp) FILTER (WHERE report_date <= DATE '2026-03-31') >= 100""").df()
fr['log_imp_mar'] = np.log1p(fr.imp_mar)
fr['ctr_mar'] = 100 * fr.clk_mar / fr.imp_mar
fr['pos_mar'] = fr.pos_mar.fillna(fr.pos_mar.median())
fr['days_with_imp_mar'] = fr.days_mar
fr['spike_share_mar'] = fr.max_day_mar / fr.imp_mar
# label (April): did the page fall behind its own site? page ratio < 0.8 x site ratio
site = fr.groupby('client_hash_id')[['imp_apr', 'imp_mar']].sum()
fr['client_ratio'] = fr.client_hash_id.map(site.imp_apr / site.imp_mar)
fr['declined'] = ((fr.imp_apr / fr.imp_mar) < 0.8 * fr.client_ratio).astype(int)
F5 = ['log_imp_mar', 'ctr_mar', 'pos_mar', 'days_with_imp_mar', 'spike_share_mar']
print('\nFIVE-FEATURE FRAME - each feature and why it is knowable on 2026-03-31:')
print(pd.DataFrame({'feature': F5, 'available when?': [
    'March impressions are final once March 31 ends',
    'March clicks / March impressions - same days, same cut-off',
    'impression-weighted rank over March days only',
    'count of March days with impressions - nothing after March 31',
    'biggest March day / March total - March only']}).to_string(index=False))
print(f'\nframe: {len(fr):,} pages, {fr.client_hash_id.nunique()} clients, April decline rate {fr.declined.mean():.3f}')

# ---- The trap: one label-derived column on purpose, then delete it ----
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score
fr['fold'] = fr.client_hash_id.map(ff.client_fold)
def quick_auc(cols):
    # client-grouped: every page is scored by a model that never saw its client
    s = pd.Series(0.0, index=fr.index)
    for f in range(4):
        tr, te = fr.fold != f, fr.fold == f
        mdl = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)).fit(fr.loc[tr, cols], fr.loc[tr, 'declined'])
        s[te] = mdl.predict_proba(fr.loc[te, cols])[:, 1]
    return roc_auc_score(fr.declined, s)
honest = quick_auc(F5)
fr['leak_log_imp_apr'] = np.log1p(fr.imp_apr)          # April impressions = the outcome month itself
leaky = quick_auc(F5 + ['leak_log_imp_apr'])
print(f'\nquick score, honest 5 features:            AUC {honest:.3f}')
print(f'quick score, + one label-derived column:    AUC {leaky:.3f}   <- too good to be true')
fr = fr.drop(columns='leak_log_imp_apr')
print('leak column deleted:', 'leak_log_imp_apr' not in fr.columns, f'| the number I keep: AUC {quick_auc(F5):.3f}')

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

**Named limitation: an unbalanced panel.** Clients joined at very different times. Only 40-41 of the 104 clients have a GSC feed covering the full feature window at the two snapshots, and after the other checks the model population is 25-27 clients, not 104. Results describe those clients.

Other things this slice can't tell me:

- **Early GSC-only history.** GA4 starts in February/March 2026 for most clients, so engagement can't be a feature without throwing away most of the history.
- **Query table overlap.** It covers April-June 2026 only, which is the same as my outcome months, so it is excluded.
- **Edits after T.** Most panel rows belong to pages edited *after* the decision date (see `updated_after_T`). I can't see the page as it was on day T, only its metrics.
- **Site-wide swings.** The median site's impressions fell about 26% in May (median site ratio 0.74, against 0.89 in April). Part of the label is defined relative to that, which is a choice, not a fact of nature.
- **No causal reading.** Nothing here shows what a refresh would *do*; it only shows which pages tended to fall behind.

In [ ]:
print('clients by GSC start quarter (unbalanced panel):')
print(pd.to_datetime(cl.gsc_data_start).dt.to_period('Q').value_counts().sort_index().to_string())
q = con.sql(f"SELECT MIN(window_start) s, MAX(window_end) e FROM read_parquet('{ff.REL}/fact_content_query_90d.parquet')").fetchone()
print(f'\nquery table window: {q[0]} -> {q[1]}  (label months used: April, May, June 2026 -> overlap)')
print(f"share of panel rows whose page was edited AFTER the decision date: {panel.updated_after_T.mean():.3f}")
print(f"share of panel rows that vanish in the outcome month: {panel.vanished_next30.mean():.3f}")
print(f"share of panel rows dropped by the population checks: {1 - len(m)/len(panel):.3f}")
print('\nsite-level swing in the outcome month (median client_ratio by snapshot):', m.groupby('decision_date').client_ratio.median().round(3).to_dict())

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.